In [14]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/My Drive/Colab Notebooks')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import numpy as np
import pandas as pd
from scipy.io.arff import loadarff
import torch

In [16]:
import pandas as pd

# Load the CSV file
file_path = 'Data/ECG_200_train_sample_reservoir_states.csv'
df = pd.read_csv(file_path, header=None)

# Convert the DataFrame to a 2D NumPy array
numpy_array = df.to_numpy()

# Get the number of rows and columns
numpy_array.shape

(1600, 97)

In [17]:
reshaped_array = numpy_array.reshape(int(numpy_array.shape[0]/16), 16, numpy_array.shape[1])
tensor = torch.tensor(reshaped_array, dtype=torch.float32)

# Verify the new tensor shape
tensor.shape

torch.Size([100, 16, 97])

In [18]:
split_index = int(0.7 * tensor.shape[0])  # 80% of 861 → 688.8 → 689

train_tensor = tensor[:split_index]   # [689, 16, 137]
test_tensor = tensor[split_index:]    # [172, 16, 137]

# Print shapes to verify
print(train_tensor.shape)  # torch.Size([689, 16, 137])
print(test_tensor.shape)   # torch.Size([172, 16, 137])

torch.Size([70, 16, 97])
torch.Size([30, 16, 97])


In [19]:
num_timesteps = tensor.shape[2]-1

In [20]:
# Split the tensor into two parts
tensor_first = train_tensor[:, :, :num_timesteps]  # Contains the first 152 columns
tensor_last_column = train_tensor[:, :, num_timesteps]  # Contains the last column
tensor_last_column_reduced = tensor_last_column[:, 0]
train_x = tensor_first
train_y = tensor_last_column_reduced


In [21]:
# Split the tensor into two parts
test_tensor_first = test_tensor[:, :, :num_timesteps]  # Contains the first 152 columns
test_tensor_last_column = test_tensor[:, :, num_timesteps]  # Contains the last column
test_tensor_last_column_reduced = test_tensor_last_column[:, 0]
test_x = test_tensor_first
test_y = test_tensor_last_column_reduced

In [22]:
train_y[np.where(train_y != 1)] = 0
test_y[np.where(test_y != 1)] = 0

In [23]:
train_y

tensor([0., 1., 0., 0., 1., 1., 0., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.,
        0., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0.,
        1., 1., 1., 1., 1., 1., 1., 0., 0., 1., 0., 0., 1., 0., 1., 1., 1., 0.,
        1., 1., 0., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1.])

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim

# Sample dimensions

d = 8
n = num_timesteps
input_dim = 2 * d * n  # Flattened input size (16 * 96)


#train_x = torch.randn(batch_size, 2 * d, n)  # Shape: (999, 16, 152)
train_x = train_x.float()
#train_y = torch.randint(0, 2, (batch_size,))  # Binary labels for two classes
# Convert train_y from numpy array to torch tensor
train_y_tensor = torch.tensor(train_y, dtype=torch.long)  # Ensure the correct dtype for classification

# Define the model
class Classifier(nn.Module):
    def __init__(self, input_dim):
        super(Classifier, self).__init__()
        self.fc = nn.Linear(input_dim, 2)  # Fully connected layer for 2 classes

    def forward(self, x):
        #x = x.view(x.size(0), -1)  # Flatten 2*d x n to (batch_size, 2*d*n)
        x = x.reshape(x.size(0), -1)  # Flatten 2*d x n to (batch_size, 2*d*n)
        return self.fc(x)

# Instantiate the model
model = Classifier(input_dim)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.1, weight_decay=0.1)

# Training loop
num_epochs = 30
for epoch in range(num_epochs):
    model.train()

    # Forward pass
    outputs = model(train_x)
    loss = criterion(outputs, train_y_tensor)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/30], Loss: 0.8263
Epoch [2/30], Loss: 21.6528
Epoch [3/30], Loss: 18.5173
Epoch [4/30], Loss: 8.3620
Epoch [5/30], Loss: 0.4555
Epoch [6/30], Loss: 14.2674
Epoch [7/30], Loss: 4.8257
Epoch [8/30], Loss: 0.2752
Epoch [9/30], Loss: 4.6843
Epoch [10/30], Loss: 7.5859
Epoch [11/30], Loss: 7.2356
Epoch [12/30], Loss: 4.2759
Epoch [13/30], Loss: 0.8445
Epoch [14/30], Loss: 0.1294
Epoch [15/30], Loss: 0.5914
Epoch [16/30], Loss: 3.5560
Epoch [17/30], Loss: 2.7041
Epoch [18/30], Loss: 0.3513
Epoch [19/30], Loss: 0.0540
Epoch [20/30], Loss: 0.2289
Epoch [21/30], Loss: 0.8970
Epoch [22/30], Loss: 1.6734
Epoch [23/30], Loss: 1.2970
Epoch [24/30], Loss: 0.3222
Epoch [25/30], Loss: 0.0232
Epoch [26/30], Loss: 0.0018
Epoch [27/30], Loss: 0.2017
Epoch [28/30], Loss: 1.1693
Epoch [29/30], Loss: 0.1510
Epoch [30/30], Loss: 0.0003


<ipython-input-24-de0e3cad7ffa>:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_y_tensor = torch.tensor(train_y, dtype=torch.long)  # Ensure the correct dtype for classification


In [25]:
model.eval()
with torch.no_grad():
    # Ensure final_output is of float type


    # Perform inference
    outputs = model(train_x)  # Get the predictions
    _, predicted = torch.max(outputs, 1)  # Get the predicted class

    # Calculate accuracy
    accuracy = (predicted == train_y_tensor).sum().item() / train_y_tensor.size(0)
    print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 100.00%


In [26]:
#train_x = torch.randn(batch_size, 2 * d, n)  # Shape: (999, 16, 152)
test_x = test_x.float()
#train_y = torch.randint(0, 2, (batch_size,))  # Binary labels for two classes
# Convert train_y from numpy array to torch tensor
test_y_tensor = torch.tensor(test_y, dtype=torch.long)  # Ensure the correct dtype for classification
model.eval()
with torch.no_grad():
    # Ensure final_output is of float type


    # Perform inference
    outputs = model(test_x)  # Get the predictions
    _, predicted = torch.max(outputs, 1)  # Get the predicted class

    # Calculate accuracy
    accuracy = (predicted == test_y_tensor).sum().item() / test_y_tensor.size(0)
    print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 60.00%


<ipython-input-26-021d6be12b33>:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_y_tensor = torch.tensor(test_y, dtype=torch.long)  # Ensure the correct dtype for classification


In [28]:
predicted

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 0, 1, 1, 1, 1])